# Pizza Cluster — Full Pipeline Orchestrator

End-to-end pipeline: JMAIL raw data → preprocessing → embeddings → clustering → labelling → dataset etichettato → validazione manuale.

**Istruzioni:**
1. Impostare `DEV_MODE = True` per smoke test locale (~1k email). Verificare che tutto giri senza errori.
2. Sul server del laboratorio: `DEV_MODE = False`, `FORCE_RERUN = True` per il primo run completo.
3. Assicurarsi che `ollama serve` sia attivo prima della fase di labelling.

**Fasi:**
1. Estrazione dati raw da JMAIL
2. Preprocessing
3. Generazione embeddings (BGE-small + Token-Aware Chunking + Decay Pooling)
4. Clustering (UMAP 12D + HDBSCAN, parametri da DECISIONS.md 2026-06-19)
5. Labelling (c-TF-IDF + YAKE + TextRank + LLM naming via Ollama)
6. Dataset finale etichettato
7. File di validazione manuale


In [ ]:
import sys
from pathlib import Path

# ─── Individua la root del progetto (contiene .env) ───
def _find_root(start=Path.cwd()):
    for p in [start] + list(start.parents):
        if (p / '.env').exists():
            return p
    return start

ROOT     = _find_root()
ENV_FILE = str(ROOT / '.env')

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# ────────────────────────────────────────────────────────────
#  CONFIGURAZIONE CENTRALIZZATA — modifica qui prima di eseguire
# ────────────────────────────────────────────────────────────
DEV_MODE     = True    # True = smoke test (~1k email); False = run completo
SEED         = 42
RERUN_OPTUNA = False   # False = usa BEST_PARAMS da DECISIONS.md 2026-06-19
LLM_MODEL    = 'llama3'

# Parametri ottimali UMAP + HDBSCAN (DECISIONS.md 2026-06-19, Optuna CMA-ES su 15k)
BEST_PARAMS = {
    'n_neighbors'            : 51,
    'n_components'           : 12,
    'min_cluster_size'       : 71,
    'min_samples'            : 100,
    'cluster_selection_method': 'eom',
}

# Subsample per operazioni pesanti (O(n²))
SILHOUETTE_SAMPLE = 20_000
SCATTER_SAMPLE    = 50_000
OPTUNA_SAMPLE     = 150_000
KMEANS_K_VALUES   = [3, 5, 10, 15, 20]

# Override DEV_MODE
if DEV_MODE:
    DEV_SAMPLE_SIZE   = 1_000
    SILHOUETTE_SAMPLE = 300
    SCATTER_SAMPLE    = 500
    OPTUNA_SAMPLE     = 800
    KMEANS_K_VALUES   = [3, 5]
    FORCE_RERUN       = True   # smoke test sempre fresh
    print('[DEV_MODE] Pipeline su campione ridotto —', DEV_SAMPLE_SIZE, 'email')
else:
    DEV_SAMPLE_SIZE = None
    FORCE_RERUN     = False
    print('[FULL_MODE] Pipeline completa — set FORCE_RERUN=True al primo run')

# Output paths
FIGURES_PATH         = ROOT / 'reports' / 'figures' / 'full_pipeline'
VALIDATION_PATH      = ROOT / 'data' / 'validation'
CLUSTER_LABELS_PATH  = ROOT / 'data' / 'metadata' / 'cluster_labeling_metadata.json'
CLUSTERED_PATH       = ROOT / 'data' / 'processed' / 'jmail_emails_clustered.parquet'
CLUSTERED_SOFT_PATH  = ROOT / 'data' / 'processed' / 'jmail_emails_clustered_soft.parquet'

# Pipeline checkpoint files (intermedi, non versionati)
_CKPT_LABELS   = ROOT / 'data' / 'processed' / '_pipeline_labels.npy'
_CKPT_PROBS    = ROOT / 'data' / 'processed' / '_pipeline_probs.npy'
_CKPT_REDUCED  = ROOT / 'data' / 'processed' / '_pipeline_umap_reduced.npy'
_CKPT_SOFT     = ROOT / 'data' / 'processed' / '_pipeline_soft_membership.npy'

for _p in [FIGURES_PATH, VALIDATION_PATH, ROOT / 'data' / 'processed']:
    _p.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('ENV_FILE:', ENV_FILE)


In [ ]:
import platform, subprocess, psutil

def detect_hardware():
    info = {
        'platform'        : platform.system(),
        'cpu_physical'    : psutil.cpu_count(logical=False),
        'cpu_logical'     : psutil.cpu_count(logical=True),
        'ram_gb'          : round(psutil.virtual_memory().total / 1e9, 1),
        'gpu_name'        : None,
        'vram_gb'         : None,
    }
    try:
        r = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
            capture_output=True, text=True, timeout=10,
        )
        if r.returncode == 0:
            parts = r.stdout.strip().split('\n')[0].split(',')
            info['gpu_name'] = parts[0].strip()
            info['vram_gb']  = round(int(parts[1].strip().split()[0]) / 1024, 1)
    except Exception:
        pass
    return info

hw = detect_hardware()
print('Hardware rilevato:')
for k, v in hw.items():
    print(f'  {k}: {v}')

USE_GPU  = hw['gpu_name'] is not None
HAS_CUML = False
if USE_GPU:
    try:
        import cuml  # noqa: F401
        HAS_CUML = True
        print(f'\ncuML disponibile — UMAP/HDBSCAN su GPU ({hw["gpu_name"]})')
    except ImportError:
        print(f'\nGPU rilevata ({hw["gpu_name"]}) ma cuML non installato — fallback CPU')
        USE_GPU = False
else:
    print('\nNessuna GPU — CPU mode')


In [ ]:
import json, time, logging, warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg') if not DEV_MODE else None  # headless on lab server
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

from dotenv import dotenv_values

from src.utils.data_extraction   import run_extraction
from src.utils.data_processing   import run_processing_with_limit
from src.utils.embedding_pipeline import run_embedding_pipeline
from src.utils.optuna_clustering  import run_clustering_optimization
from src.utils.topic_labeling     import (
    calculate_ctfidf,
    extract_keywords_yake,
    extract_keywords_textrank,
    get_cluster_representative_docs,
)
from src.utils.llm_naming import get_llm_cluster_name

if HAS_CUML:
    from cuml.manifold import UMAP
    from cuml.cluster  import HDBSCAN as cuHDBSCAN
else:
    from umap import UMAP
    import hdbscan as hdbscan_lib

print('Imports OK')
print(f'pandas {pd.__version__}  numpy {np.__version__}')

_env = dotenv_values(ENV_FILE)


In [ ]:
def checkpoint_exists(*paths):
    '''True se tutti i path esistono E FORCE_RERUN è False.'''
    if FORCE_RERUN:
        return False
    return all(Path(p).exists() for p in paths)

def save_figure(fig, name: str, metadata: dict = None):
    '''Salva PNG + JSON metadati in FIGURES_PATH.'''
    png  = FIGURES_PATH / f'{name}.png'
    meta = FIGURES_PATH / f'{name}.json'
    fig.savefig(png, dpi=150, bbox_inches='tight')
    m = {'name': name, 'seed': SEED, 'dev_mode': DEV_MODE, **(metadata or {})}
    meta.write_text(json.dumps(m, indent=2, default=str), encoding='utf-8')
    print(f'  Figure: {png.name}')

def subsample(arr: np.ndarray, labels: np.ndarray, n: int, seed=SEED):
    '''Campionamento casuale per metriche pesanti.'''
    n = min(n, len(arr))
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(arr), size=n, replace=False)
    return arr[idx], labels[idx]

print('Helper OK')


## Fase 1 — Estrazione dati raw da JMAIL

In [ ]:
_raw_dir  = ROOT / _env.get('DATA_RAW_PATH', 'data/raw/')
_raw_file = _raw_dir / _env.get('RAW_EMAILS_FILENAME', 'jmail_emails.parquet')

if checkpoint_exists(_raw_file):
    print(f'[SKIP] Raw già presente: {_raw_file}')
    import pyarrow.parquet as pq
    _n_raw = pq.read_metadata(_raw_file).num_rows
    print(f'       Righe: {_n_raw:,}')
else:
    print('Scaricamento dati raw da JMAIL...')
    t0 = time.time()
    _extr = run_extraction(ENV_FILE, sample_limit=DEV_SAMPLE_SIZE if DEV_MODE else None)
    print(f'Estrazione completata in {time.time()-t0:.1f}s')
    print(f'  Righe:  {_extr["row_count"]:,}')
    print(f'  Output: {_extr["raw_output_path"]}')


## Fase 2 — Preprocessing

In [ ]:
_proc_dir  = ROOT / _env.get('DATA_PROCESSED_PATH', 'data/processed/')
_proc_file = _proc_dir / _env.get('PROCESSED_EMAILS_FILENAME', 'jmail_emails_processed.parquet')

# In DEV_MODE, run_processing_with_limit(limit=N) scrive su PROCESSED_EMAILS_SAMPLE_FILENAME
_proc_file_dev = _proc_dir / _env.get('PROCESSED_EMAILS_SAMPLE_FILENAME', 'jmail_emails_processed.parquet')
_active_proc   = _proc_file_dev if DEV_MODE else _proc_file

if checkpoint_exists(_active_proc):
    print(f'[SKIP] Processed già presente: {_active_proc}')
    df_processed = pd.read_parquet(_active_proc)
else:
    print('Avvio preprocessing...')
    t0 = time.time()
    _res = run_processing_with_limit(ENV_FILE, limit=DEV_SAMPLE_SIZE if DEV_MODE else None)
    df_processed = pd.read_parquet(_active_proc)
    print(f'Preprocessing completato in {time.time()-t0:.1f}s')
    print(f'  Input:              {_res["input_rows"]:,}')
    print(f'  Output:             {_res["output_rows"]:,}')
    print(f'  Rimossi promo:      {_res["removed_promotional_rows"]:,}')
    print(f'  Rimossi testo vuoto:{_res["removed_empty_text_rows"]:,}')

print(f'\nDataset processato: {df_processed.shape}')
df_processed[['combined_text', 'has_thread', 'has_redaction', 'sender_domain']].head(3)


In [ ]:
# ─── Analisi raw vs processed ───
_raw_df = pd.read_parquet(_raw_file)

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Raw vs Processed — Overview', fontsize=14, fontweight='bold')

# 1. Dataset size
ax = axes[0, 0]
_sizes = [len(_raw_df), len(df_processed)]
_bars  = ax.bar(['Raw', 'Processed'], _sizes, color=['#4C72B0', '#55A868'])
ax.bar_label(_bars, fmt='{:,.0f}', padding=3, fontsize=10)
ax.set_title('Numero email')
ax.set_ylabel('Count')

# 2. Distribuzione lunghezza combined_text
ax = axes[0, 1]
df_processed['combined_text_length'].clip(upper=5000).hist(bins=50, ax=ax, color='#4C72B0', alpha=0.8)
ax.set_title('Lunghezza combined_text (clip 5k car.)')
ax.set_xlabel('Caratteri')
ax.set_ylabel('Frequenza')

# 3. Flag diagnostici
ax = axes[1, 0]
_flags = {
    'has_thread'    : int(df_processed['has_thread'].sum()),
    'has_redaction' : int(df_processed['has_redaction'].sum()),
    'has_disclaimer': int(df_processed['has_disclaimer'].sum()),
    'person_unknown': int(df_processed['person_unknown'].sum()),
}
ax.barh(list(_flags.keys()), list(_flags.values()), color='#4C72B0', alpha=0.8)
ax.set_title('Flag diagnostici')
ax.set_xlabel('Count')

# 4. Null ratio
ax = axes[1, 1]
_cols   = ['subject_clean', 'content_clean', 'sender_domain', 'sent_at_datetime']
_nulls  = [df_processed[c].isnull().mean() for c in _cols]
ax.bar(_cols, _nulls, color='#C44E52', alpha=0.8)
ax.set_title('Null ratio colonne chiave')
ax.set_ylabel('Ratio (0–1)')
ax.set_ylim(0, 1)
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')

plt.tight_layout()
save_figure(fig, '01_raw_vs_processed', {'raw_rows': len(_raw_df), 'processed_rows': len(df_processed)})
plt.show()

# Tabella shape
print('\n── Shape ──')
print(pd.DataFrame({
    'dataset': ['raw', 'processed'],
    'rows'   : [len(_raw_df), len(df_processed)],
    'cols'   : [len(_raw_df.columns), len(df_processed.columns)],
}).to_string(index=False))

# 5 con thread + 5 senza
print('\n── Campione 10 email (5 con thread, 5 senza) ──')
_s = pd.concat([
    df_processed[df_processed['has_thread']].head(5),
    df_processed[~df_processed['has_thread']].head(5),
])
print(_s[['id', 'has_thread', 'combined_text']].assign(
    preview=lambda d: d['combined_text'].str[:80] + '...'
)[['id', 'has_thread', 'preview']].to_string(index=False))


## Fase 3 — Generazione embeddings

Modello: `BAAI/bge-small-en-v1.5` con Token-Aware Chunking (400 token, overlap 15%) e Weighted Decay Pooling (DECISIONS.md 2026-06-17).

In [ ]:
_emb_dir   = ROOT / _env.get('DATA_EMBEDDINGS_PATH', 'data/embeddings/')
_emb_file  = _emb_dir / _env.get('EMBEDDINGS_FILENAME', 'email_embeddings.npy')
_idx_file  = ROOT / _env.get('METADATA_PATH', 'data/metadata/') / _env.get('EMBEDDING_INDEX_FILENAME', 'email_embedding_index.parquet')

if checkpoint_exists(_emb_file, _idx_file):
    print(f'[SKIP] Embeddings già presenti: {_emb_file}')
else:
    print(f'Generazione embeddings (modello: {_env.get("EMBEDDING_MODEL_NAME", "bge-small")})...')
    t0 = time.time()
    _emb_meta = run_embedding_pipeline(ENV_FILE)
    print(f'Completato in {time.time()-t0:.1f}s')
    print(f'  Shape: {_emb_meta["row_count"]} × {_emb_meta["embedding_dimensions"]}')
    print(f'  Chunk: {_emb_meta["chunk_count"]}')

embeddings      = np.load(_emb_file)
embedding_index = pd.read_parquet(_idx_file)
print(f'\nEmbeddings caricati: {embeddings.shape}')


In [ ]:
# ─── Diagnostica embeddings ───
assert len(embeddings) == len(df_processed), (
    f'MISMATCH: {len(embeddings)} embeddings vs {len(df_processed)} email processate. '
    'Rigenerare gli embeddings (FORCE_RERUN=True nella sezione Fase 3).'
)

norms      = np.linalg.norm(embeddings, axis=1)
norm_range = norms.max() - norms.min()
n_bins     = 1 if norm_range < 1e-9 else 30

_chunk_counts = embedding_index.groupby('embedding_row').size() if 'embedding_row' in embedding_index.columns else pd.Series([1] * len(embeddings))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(norms, bins=n_bins, color='#4C72B0', alpha=0.8)
axes[0].set_title('Distribuzione norme L2')
axes[0].set_xlabel('Norma L2')

axes[1].hist(_chunk_counts, bins=min(30, int(_chunk_counts.max())), color='#55A868', alpha=0.8)
axes[1].set_title('Chunk per email')
axes[1].set_xlabel('N chunk')

plt.tight_layout()
save_figure(fig, '02_embedding_diagnostics', {'shape': list(embeddings.shape)})
plt.show()

print(f'Embedding dim:  {embeddings.shape[1]}')
print(f'Norma media:    {norms.mean():.4f}')
print(f'Norma std:      {norms.std():.6f}')
print(f'Chunk totali:   {len(embedding_index)}')


In [ ]:
# ─── UMAP 2D per visualizzazione (campione) ───
_sc_n   = min(SCATTER_SAMPLE, len(embeddings))
_rng    = np.random.default_rng(SEED)
_sc_idx = _rng.choice(len(embeddings), size=_sc_n, replace=False)
_emb_sc = embeddings[_sc_idx]

print(f'UMAP 2D su {_sc_n} punti (solo visualizzazione)...')
t0 = time.time()
_reducer2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine', random_state=SEED) \
    if HAS_CUML else \
    UMAP(n_components=2, n_neighbors=30, min_dist=0.1, metric='cosine', random_state=SEED, low_memory=True)
_emb2d = _reducer2d.fit_transform(_emb_sc)
print(f'  Completato in {time.time()-t0:.1f}s')

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(_emb2d[:, 0], _emb2d[:, 1], s=2, alpha=0.4, color='#4C72B0', rasterized=True)
ax.set_title(f'UMAP 2D — {_sc_n} email (campione)')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
save_figure(fig, '03_umap_2d_scatter', {'sample_size': _sc_n})
plt.show()


In [ ]:
# ─── MiniBatchKMeans sweep ───
_sil_n   = min(SILHOUETTE_SAMPLE, len(embeddings))
_sil_rng = np.random.default_rng(SEED)
_sil_idx = _sil_rng.choice(len(embeddings), size=_sil_n, replace=False)
_emb_sil = embeddings[_sil_idx]

_km_rows = []
print(f'MiniBatchKMeans sweep K={KMEANS_K_VALUES} su {_sil_n} campioni...')
for k in KMEANS_K_VALUES:
    _km    = MiniBatchKMeans(n_clusters=k, random_state=SEED, n_init=3, batch_size=min(10_000, _sil_n))
    _lab_k = _km.fit_predict(_emb_sil)
    _sil   = silhouette_score(_emb_sil, _lab_k, sample_size=min(5_000, _sil_n), random_state=SEED)
    _db    = davies_bouldin_score(_emb_sil, _lab_k)
    _ch    = calinski_harabasz_score(_emb_sil, _lab_k)
    _km_rows.append({'K': k, 'Silhouette': round(_sil, 4), 'Davies-Bouldin': round(_db, 4), 'Calinski-Harabasz': round(_ch, 1)})
    print(f'  K={k:2d}: Sil={_sil:.4f}  DB={_db:.4f}  CH={_ch:.1f}')

df_kmeans = pd.DataFrame(_km_rows)
print('\n── MiniBatchKMeans metrics ──')
print(df_kmeans.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Silhouette', 'Davies-Bouldin', 'Calinski-Harabasz']):
    ax.plot(df_kmeans['K'], df_kmeans[col], marker='o', color='#4C72B0')
    ax.set_title(col)
    ax.set_xlabel('K')
plt.tight_layout()
save_figure(fig, '04_kmeans_sweep', {'k_values': KMEANS_K_VALUES, 'sample_size': _sil_n})
plt.show()


## Fase 4 — Clustering (UMAP 12D + HDBSCAN)

Parametri da `DECISIONS.md` 2026-06-19 (Optuna CMA-ES su 15k sample, Silhouette ~0.63):
- `n_neighbors=51`, `n_components=12`, `min_cluster_size=71`, `min_samples=100`, `cluster_selection_method='eom'`

Strategia di scala: UMAP + HDBSCAN girano sull'intero dataset. Optuna (se `RERUN_OPTUNA=True`) gira su sottocampione e rifissa i best params sul full.


In [ ]:
# ─── Optuna (opzionale) ───
if RERUN_OPTUNA:
    _opt_n   = min(OPTUNA_SAMPLE, len(embeddings))
    _opt_rng = np.random.default_rng(SEED)
    _opt_idx = _opt_rng.choice(len(embeddings), size=_opt_n, replace=False)
    _emb_opt = embeddings[_opt_idx]
    print(f'Optuna CMA-ES su {_opt_n} campioni (n_trials=30)...')
    _study    = run_clustering_optimization(_emb_opt, n_trials=30, use_umap=True, sampler_type='CMA-ES')
    best_params = _study.best_params
    print(f'Best params trovati: {best_params}')
else:
    best_params = BEST_PARAMS
    print('Usando parametri da DECISIONS.md 2026-06-19:')
    for k, v in best_params.items():
        print(f'  {k}: {v}')


In [ ]:
# ─── UMAP 12D + HDBSCAN sul dataset completo ───
if checkpoint_exists(_CKPT_LABELS, _CKPT_PROBS, _CKPT_REDUCED):
    print('[SKIP] Checkpoint clustering già presente — caricamento...')
    labels        = np.load(_CKPT_LABELS)
    cluster_probs = np.load(_CKPT_PROBS)
    emb_reduced   = np.load(_CKPT_REDUCED)
    print(f'  labels shape:      {labels.shape}')
    print(f'  emb_reduced shape: {emb_reduced.shape}')
else:
    _nn  = best_params.get('n_neighbors', 51)
    _nc  = best_params.get('n_components', 12)
    _mcs = best_params.get('min_cluster_size', 71)
    _ms  = best_params.get('min_samples', 100)
    _csm = best_params.get('cluster_selection_method', 'eom')

    print(f'UMAP {_nc}D su {len(embeddings):,} vettori (metric=cosine, low_memory su CPU)...')
    t0 = time.time()
    if HAS_CUML:
        _reducer = UMAP(n_components=_nc, n_neighbors=_nn, metric='cosine', min_dist=0.01, random_state=SEED)
    else:
        _reducer = UMAP(n_components=_nc, n_neighbors=_nn, metric='cosine', min_dist=0.01,
                        random_state=SEED, low_memory=True)
    emb_reduced = np.array(_reducer.fit_transform(embeddings))
    print(f'  UMAP completato in {time.time()-t0:.1f}s  shape={emb_reduced.shape}')

    print(f'HDBSCAN (min_cluster_size={_mcs}, min_samples={_ms})...')
    t0 = time.time()
    if HAS_CUML:
        _clust = cuHDBSCAN(min_cluster_size=_mcs, min_samples=_ms,
                           cluster_selection_method=_csm)
        _clust.fit(emb_reduced)
        labels        = np.array(_clust.labels_.get() if hasattr(_clust.labels_, 'get') else _clust.labels_)
        cluster_probs = np.array(_clust.probabilities_.get() if hasattr(_clust.probabilities_, 'get') else _clust.probabilities_)
    else:
        _clust = hdbscan_lib.HDBSCAN(
            min_cluster_size=_mcs, min_samples=_ms,
            metric='euclidean', cluster_selection_method=_csm,
            prediction_data=True, core_dist_n_jobs=-1,
        )
        _clust.fit(emb_reduced)
        labels        = _clust.labels_
        cluster_probs = _clust.probabilities_
    print(f'  HDBSCAN completato in {time.time()-t0:.1f}s')

    # Salva checkpoint
    np.save(_CKPT_LABELS,  labels)
    np.save(_CKPT_PROBS,   cluster_probs)
    np.save(_CKPT_REDUCED, emb_reduced)
    print('  Checkpoint salvato.')

n_clusters  = len(set(labels)) - (1 if -1 in labels else 0)
noise_ratio = float((labels == -1).mean())
print(f'\nCluster trovati: {n_clusters}')
print(f'Noise ratio:     {noise_ratio:.2%}')


In [ ]:
# ─── Metriche clustering ───
_mask_cl  = labels != -1
_emb_cl   = emb_reduced[_mask_cl]
_lab_cl   = labels[_mask_cl]
_emb_m, _lab_m = subsample(_emb_cl, _lab_cl, SILHOUETTE_SAMPLE)

sil = silhouette_score(_emb_m, _lab_m)
db  = davies_bouldin_score(_emb_m, _lab_m)
ch  = calinski_harabasz_score(_emb_m, _lab_m)

print('── Metriche clustering finale ──')
print(f'  Silhouette Score:        {sil:.4f}')
print(f'  Davies-Bouldin Score:    {db:.4f}')
print(f'  Calinski-Harabasz Score: {ch:.1f}')
print(f'  N cluster:               {n_clusters}')
print(f'  Noise ratio:             {noise_ratio:.2%}')
print(f'  Email clusterizzate:     {_mask_cl.sum():,}  ({_mask_cl.mean():.1%})')


In [ ]:
# ─── Scatter 2D colorato per cluster ───
_sc_n   = min(SCATTER_SAMPLE, len(emb_reduced))
_sc_rng = np.random.default_rng(SEED)
_sc_idx = _sc_rng.choice(len(emb_reduced), size=_sc_n, replace=False)

print(f'UMAP 2D (su reduced 12D → 2D) per {_sc_n} punti...')
t0 = time.time()
_red2d = UMAP(n_components=2, n_neighbors=30, min_dist=0.05, random_state=SEED) \
    if HAS_CUML else \
    UMAP(n_components=2, n_neighbors=30, min_dist=0.05, random_state=SEED, low_memory=True)
_emb_2d_cl = np.array(_red2d.fit_transform(emb_reduced[_sc_idx]))
_lab_2d_cl = labels[_sc_idx]
print(f'  completato in {time.time()-t0:.1f}s')

_unique = sorted(set(_lab_2d_cl))
_cmap   = cm.get_cmap('tab20', max(len(_unique), 1))

fig, ax = plt.subplots(figsize=(13, 10))
for i, cl in enumerate(_unique):
    _m     = _lab_2d_cl == cl
    _color = 'lightgray' if cl == -1 else _cmap(i % 20)
    _alpha = 0.2 if cl == -1 else 0.5
    ax.scatter(_emb_2d_cl[_m, 0], _emb_2d_cl[_m, 1], s=2, alpha=_alpha,
               color=_color, rasterized=True, label=str(cl) if cl != -1 else 'noise')
ax.set_title(f'Cluster scatter 2D — {_sc_n:,} email, {n_clusters} cluster')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
save_figure(fig, '05_cluster_scatter_2d', {
    'n_clusters': n_clusters, 'noise_ratio': noise_ratio, 'sample_size': _sc_n
})
plt.show()


In [ ]:
# ─── Soft clustering + overlap network ───
_has_soft = (not HAS_CUML) and '_clust' in dir() and hasattr(_clust, 'all_points_membership_vectors')
_soft_checkpoint = checkpoint_exists(_CKPT_SOFT)

if _has_soft or _soft_checkpoint:
    if _soft_checkpoint:
        print('[SKIP] Soft membership già presente — caricamento...')
        _membership = np.load(_CKPT_SOFT)
    else:
        print('Calcolo soft membership vectors...')
        try:
            _membership = hdbscan_lib.all_points_membership_vectors(_clust)
            np.save(_CKPT_SOFT, _membership)
            print(f'  Shape: {_membership.shape}  Checkpoint salvato.')
        except Exception as e:
            print(f'  Soft clustering non disponibile: {e}')
            _membership = None

    if _membership is not None:
        _top10_idx   = np.argsort(_membership, axis=1)[:, ::-1][:, :10]
        _top10_probs = np.sort(_membership, axis=1)[:, ::-1][:, :10]

        _df_soft = df_processed.copy()
        _df_soft['cluster']       = labels
        _df_soft['cluster_prob']  = cluster_probs
        _df_soft['top_10_clusters'] = list(_top10_idx)
        _df_soft['top_10_probs']    = list(_top10_probs)
        _df_soft.to_parquet(CLUSTERED_SOFT_PATH, index=False)
        print(f'Soft clustering salvato: {CLUSTERED_SOFT_PATH}')

        # Network graph a livello di cluster (nodi = cluster, archi = overlap > soglia)
        try:
            import networkx as nx
            _OVERLAP_THR = 0.05
            _cids = [c for c in sorted(set(labels)) if c != -1]
            G = nx.Graph()
            G.add_nodes_from(_cids)
            for _i, _ci in enumerate(_cids):
                _probs_i = _membership[labels == _ci]
                for _j, _cj in enumerate(_cids):
                    if _j <= _i:
                        continue
                    _ov = float(_probs_i[:, _cj].mean())
                    if _ov > _OVERLAP_THR:
                        G.add_edge(_ci, _cj, weight=_ov)

            fig, ax = plt.subplots(figsize=(13, 11))
            _pos  = nx.spring_layout(G, seed=SEED, k=2)
            _ews  = [G[u][v]['weight'] * 5 for u, v in G.edges()]
            nx.draw_networkx(G, pos=_pos, ax=ax, node_size=300, font_size=8,
                             edge_color='#999', width=_ews, alpha=0.85)
            ax.set_title(f'Cluster overlap network (soglia overlap > {_OVERLAP_THR:.0%})')
            ax.axis('off')
            save_figure(fig, '06_cluster_overlap_network', {'overlap_threshold': _OVERLAP_THR})
            plt.show()
        except ImportError:
            print('networkx non installato — network graph saltato. Installare con: uv pip install networkx')
else:
    print('Soft clustering disponibile solo con hdbscan CPU (prediction_data=True). Saltato.')


## Fase 5 — Labelling

Estrazione keyword con 3 algoritmi (c-TF-IDF, YAKE, TextRank), poi naming tramite LLM locale (Ollama/llama3).

> **Nota:** KeyBERT è omesso perché richiede word embeddings precomputati per ogni parola del vocabolario — costo proibitivo su 1.7M email. I 3 algoritmi coprono semantica statistica (TF-IDF), rilevanza locale (YAKE) e struttura del grafo (TextRank).


In [ ]:
_cluster_ids = sorted(c for c in set(labels) if c != -1)
print(f'Estrazione keyword per {len(_cluster_ids)} cluster...')

# Documenti aggregati per cluster
_docs_per_cluster = {}
for cid in _cluster_ids:
    _mask = labels == cid
    _texts = df_processed.loc[_mask, 'combined_text'].fillna('').tolist()
    _docs_per_cluster[cid] = ' '.join(_texts)

# c-TF-IDF
print('  c-TF-IDF...')
ctfidf_kw = calculate_ctfidf(_docs_per_cluster, top_n=20)

# YAKE
print('  YAKE...')
yake_kw = {}
for cid, doc in _docs_per_cluster.items():
    try:
        yake_kw[cid] = extract_keywords_yake(doc[:50_000], top_n=20)
    except Exception as e:
        yake_kw[cid] = []
        logging.warning(f'YAKE cluster {cid}: {e}')

# TextRank
print('  TextRank...')
textrank_kw = {}
for cid, doc in _docs_per_cluster.items():
    try:
        textrank_kw[cid] = extract_keywords_textrank(doc[:30_000], top_n=20)
    except Exception as e:
        textrank_kw[cid] = []
        logging.warning(f'TextRank cluster {cid}: {e}')

print('Estrazione keyword completata.')


In [ ]:
# ─── Tabella keyword top 5 per algoritmo ───
_kw_rows = []
for cid in _cluster_ids:
    _kw_rows.append({
        'cluster'   : cid,
        'n_email'   : int((labels == cid).sum()),
        'ctfidf'    : ', '.join(ctfidf_kw.get(cid, [])[:5]),
        'yake'      : ', '.join(yake_kw.get(cid, [])[:5]),
        'textrank'  : ', '.join(textrank_kw.get(cid, [])[:5]),
    })
df_keywords = pd.DataFrame(_kw_rows)
print('── Keyword (top 5 per algoritmo) ──')
print(df_keywords.to_string(index=False, max_colwidth=45))


In [ ]:
# ─── LLM naming via Ollama ───
# Assicurarsi che `ollama serve` sia attivo
cluster_names = {}
print(f'LLM naming su {len(_cluster_ids)} cluster (modello: {LLM_MODEL})...')
print('Verificare che Ollama sia avviato (ollama serve)\n')

for cid in _cluster_ids:
    # Unione e dedup keyword dai 3 algoritmi
    _combined = (
        ctfidf_kw.get(cid, [])[:7] +
        yake_kw.get(cid, [])[:7] +
        textrank_kw.get(cid, [])[:7]
    )
    _seen, _unique = set(), []
    for _k in _combined:
        if _k.lower() not in _seen:
            _seen.add(_k.lower())
            _unique.append(_k)

    _name = get_llm_cluster_name(_unique[:20], model=LLM_MODEL)
    cluster_names[cid] = _name
    print(f'  [{cid:3d}] {_name}')

cluster_names[-1] = 'Outlier'
print('\nNaming completato.')

# Salva metadata labelling (DATA_CONTRACTS — cluster_labeling_metadata.json)
_label_meta = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'n_clusters'    : len(_cluster_ids),
    'llm_model'     : LLM_MODEL,
    'seed'          : SEED,
    'clusters'      : {
        str(cid): {
            'n_email' : int((labels == cid).sum()),
            'ctfidf'  : ctfidf_kw.get(cid, []),
            'yake'    : yake_kw.get(cid, []),
            'textrank': textrank_kw.get(cid, []),
            'llm_name': cluster_names.get(cid, ''),
        }
        for cid in _cluster_ids
    },
}
CLUSTER_LABELS_PATH.parent.mkdir(parents=True, exist_ok=True)
CLUSTER_LABELS_PATH.write_text(json.dumps(_label_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Metadata salvati: {CLUSTER_LABELS_PATH}')


## Fase 6 — Dataset finale etichettato

Output: `data/processed/jmail_emails_clustered.parquet` (schema da `DATA_CONTRACTS.md`).

In [ ]:
if checkpoint_exists(CLUSTERED_PATH):
    print(f'[SKIP] Dataset clusterizzato già presente: {CLUSTERED_PATH}')
    df_clustered = pd.read_parquet(CLUSTERED_PATH)
else:
    df_clustered = df_processed.copy()
    df_clustered['cluster']      = labels
    df_clustered['cluster_prob'] = cluster_probs
    df_clustered['cluster_name'] = df_clustered['cluster'].map(cluster_names).fillna('Outlier')

    df_clustered.to_parquet(CLUSTERED_PATH, index=False)
    print(f'Dataset salvato: {CLUSTERED_PATH}')

print(f'\nShape finale: {df_clustered.shape}')
print('\n── Distribuzione cluster (top 15) ──')
_dist = (
    df_clustered.groupby(['cluster', 'cluster_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(15)
)
print(_dist.to_string(index=False))


## Fase 7 — Validazione manuale

Output: `data/validation/all_label.md` + `data/validation/_{{id}}/mail.md` + `label.md`.

In [ ]:
# ─── all_label.md ───
_all_label = VALIDATION_PATH / 'all_label.md'
_lines = [
    '# Cluster Labels\n\n',
    f'Generato: {datetime.now(timezone.utc).isoformat()}\n',
    f'SEED: {SEED} | LLM: {LLM_MODEL}\n\n',
    '| Cluster | Nome | N email |\n',
    '|---------|------|---------|\n',
]
for cid in sorted(cluster_names.keys()):
    _n = int((df_clustered['cluster'] == cid).sum())
    _lines.append(f'| {cid} | {cluster_names[cid]} | {_n} |\n')

_all_label.write_text(''.join(_lines), encoding='utf-8')
print(f'Salvato: {_all_label}')
print(''.join(_lines[:8]))


In [ ]:
# ─── 100 email casuali con seed fisso ───
_n_val    = min(100, len(df_clustered))
_sample   = df_clustered.sample(n=_n_val, random_state=SEED)
print(f'Generazione {_n_val} cartelle in {VALIDATION_PATH}...')

for _, row in _sample.iterrows():
    _eid     = str(row.get('id', row.name))
    _edir    = VALIDATION_PATH / f'_{_eid}'
    _edir.mkdir(parents=True, exist_ok=True)

    # mail.md — solo contenuto
    _mail  = f'# Email {_eid}\n\n'
    _mail += f'**Date:** {row.get("sent_at", "N/A")}\n'
    _mail += f'**From:** {row.get("sender", "N/A")}\n'
    _mail += f'**To:** {row.get("to_recipients", "N/A")}\n'
    _mail += f'**Subject:** {row.get("subject", "N/A")}\n\n---\n\n'
    _mail += str(row.get('content_clean', row.get('combined_text', '')))
    (_edir / 'mail.md').write_text(_mail, encoding='utf-8')

    # label.md — etichette
    _label  = f'# Label — Email {_eid}\n\n'
    _label += f'**Cluster ID:** {row.get("cluster", "N/A")}\n'
    _label += f'**Cluster Name:** {row.get("cluster_name", "N/A")}\n'
    _label += f'**Probability:** {row.get("cluster_prob", 0):.4f}\n'
    (_edir / 'label.md').write_text(_label, encoding='utf-8')

print(f'\nGenerazione completata.')
print(f'  all_label.md: {_all_label}')
print(f'  Cartelle:     {VALIDATION_PATH}/_{{id}}/mail.md + label.md')


In [ ]:
print('=' * 55)
print('PIPELINE COMPLETATA')
print('=' * 55)
_summary = {
    'mode'              : 'DEV' if DEV_MODE else 'FULL',
    'seed'              : SEED,
    'n_emails_processed': len(df_processed),
    'n_embeddings'      : len(embeddings),
    'embedding_dim'     : embeddings.shape[1],
    'n_clusters'        : n_clusters,
    'noise_ratio'       : f'{noise_ratio:.2%}',
    'silhouette'        : round(sil, 4),
    'davies_bouldin'    : round(db, 4),
    'calinski_harabasz' : round(ch, 1),
    'llm_model'         : LLM_MODEL,
    'output_clustered'  : str(CLUSTERED_PATH),
    'output_validation' : str(VALIDATION_PATH),
    'output_figures'    : str(FIGURES_PATH),
}
for k, v in _summary.items():
    print(f'  {k:<25}: {v}')
print('=' * 55)
